# Step 0 — Spatial joins with real point data and Census boundaries

This lesson assigns real point-location records to four 2024 Census geography types: counties, census tracts, incorporated places, and Census designated places (CDPs). The same pattern applies to facilities, observations, customers, assets, incidents, or other point data.

You will learn to:

1. validate coordinates and numeric measures before creating geometry;
2. align coordinate reference systems before a spatial operation;
3. use `within` for the primary point-in-polygon join;
4. retry unmatched boundary-edge points with `intersects`;
5. interpret coverage differences between statewide and settlement geographies;
6. aggregate point counts, employees, and revenue by geography and industry code; and
7. export assignments, summaries, QA tables, and map-ready geometry.

The point coordinates and measures are real data included with project authorization. When adapting this lesson, confirm that you have permission to share the source locations and attributes.

# Step 1 — Understand the two geometry roles

A spatial join combines records by location instead of a shared text key:

- the **left/input layer** is a table of point coordinates;
- the **right/join layer** is a polygon boundary file; and
- the result keeps each point and adds the identifier of the polygon containing it.

Counties and tracts cover the state, so nearly every in-state point should match them. Incorporated places and CDPs describe named settlements but do **not** cover the entire state. A point outside those polygons is an expected nonmatch, not automatically a data error.

In [1]:
# Step 2: Import the analysis and mapping libraries.

import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore", message="urllib3 .* doesn't match a supported version.*")

import folium
import geopandas as gpd
import pandas as pd
from branca.colormap import linear
from IPython.display import display
from shapely import make_valid

print(f"Python:    {sys.version.split()[0]}")
print(f"pandas:    {pd.__version__}")
print(f"GeoPandas: {gpd.__version__}")
print(f"Folium:    {folium.__version__}")

Python:    3.13.5
pandas:    3.0.1
GeoPandas: 1.1.3
Folium:    0.20.0


In [2]:
# Step 3: Define folder-relative input and output paths.

# Start Jupyter from the folder containing this notebook. Package-relative paths make
# the lesson portable and avoid assumptions about a user's machine or project layout.
LESSON_ROOT = Path.cwd().resolve()
DATA_DIR = LESSON_ROOT / "data"
OUTPUT_DIR = LESSON_ROOT / "outputs"

POINTS_CSV = DATA_DIR / "point_locations.csv"
BOUNDARY_PATHS = {
    "county": DATA_DIR / "counties_2024.geojson",
    "census_tract": DATA_DIR / "census_tracts_2024.geojson",
    "incorporated_place": DATA_DIR / "incorporated_places_2024.geojson",
    "census_designated_place": DATA_DIR / "census_designated_places_2024.geojson",
}

required_inputs = [POINTS_CSV, *BOUNDARY_PATHS.values()]
missing_inputs = [path.name for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing tutorial inputs: " + ", ".join(missing_inputs)
        + ". Start Jupyter from the folder containing this notebook and its data folder."
    )

print("All tutorial inputs are available.")
print(f"Input folder:  {DATA_DIR.name}")
print(f"Output folder: {OUTPUT_DIR.name}")

All tutorial inputs are available.
Input folder:  data
Output folder: outputs


In [3]:
# Step 4: Load the point table and inspect its schema without displaying exact locations.

string_fields = ["point_id", "industry_code", "industry_sector", "industry_description"]
points_df = pd.read_csv(POINTS_CSV, dtype={field: "string" for field in string_fields})
required_point_fields = {
    "point_id", "industry_code", "industry_sector", "industry_description",
    "employee_count", "revenue", "longitude", "latitude",
}
missing_point_fields = sorted(required_point_fields - set(points_df.columns))
if missing_point_fields:
    raise ValueError(f"Point table is missing required fields: {missing_point_fields}")

schema_summary = pd.DataFrame({
    "field": list(points_df.columns),
    "dtype": [str(points_df[field].dtype) for field in points_df.columns],
    "missing_values": [int(points_df[field].isna().sum()) for field in points_df.columns],
})
assert points_df["point_id"].notna().all() and points_df["point_id"].is_unique
assert points_df["industry_code"].notna().all()
print(f"Rows loaded: {len(points_df):,}")
display(schema_summary)

Rows loaded: 62,749


,field,dtype,missing_values
0,point_id,string,0
1,industry_code,string,0
2,industry_sector,string,0
3,industry_description,string,0
4,employee_count,int64,0
5,revenue,float64,21951
6,longitude,float64,202
7,latitude,float64,202


In [4]:
# Step 5: Validate coordinates, employee counts, and revenue.

for field in ["longitude", "latitude", "employee_count", "revenue"]:
    points_df[field] = pd.to_numeric(points_df[field], errors="coerce")

valid_coordinate_mask = (
    points_df["longitude"].between(-180, 180)
    & points_df["latitude"].between(-90, 90)
)
coordinate_valid_df = points_df.loc[valid_coordinate_mask].copy()
coordinate_issue_df = points_df.loc[~valid_coordinate_mask].copy()

# Negative employees or revenue would make summed outputs misleading, so stop if found.
assert not points_df["employee_count"].lt(0).any()
assert not points_df["revenue"].lt(0).any()

input_qa = pd.DataFrame([{
    "input_rows": len(points_df),
    "coordinate_valid_rows": len(coordinate_valid_df),
    "coordinate_issue_rows": len(coordinate_issue_df),
    "employee_value_rows": int(points_df["employee_count"].notna().sum()),
    "revenue_value_rows": int(points_df["revenue"].notna().sum()),
    "revenue_missing_rows": int(points_df["revenue"].isna().sum()),
}])
assert len(coordinate_valid_df) + len(coordinate_issue_df) == len(points_df)
assert (len(points_df), len(coordinate_valid_df), len(coordinate_issue_df)) == (62_749, 62_547, 202)
display(input_qa)

,input_rows,coordinate_valid_rows,coordinate_issue_rows,employee_value_rows,revenue_value_rows,revenue_missing_rows
0,62749,62547,202,62749,40798,21951


In [5]:
# Step 6: Create point geometry from X=longitude and Y=latitude.

point_fields = [
    "point_id", "industry_code", "industry_sector", "industry_description",
    "employee_count", "revenue", "longitude", "latitude",
]
points_gdf = gpd.GeoDataFrame(
    coordinate_valid_df[point_fields].copy(),
    geometry=gpd.points_from_xy(coordinate_valid_df["longitude"], coordinate_valid_df["latitude"]),
    crs="EPSG:4326",
)

min_x, min_y, max_x, max_y = points_gdf.total_bounds
extent_summary = pd.DataFrame([{
    "minimum_longitude": min_x, "minimum_latitude": min_y,
    "maximum_longitude": max_x, "maximum_latitude": max_y,
}])
assert points_gdf.crs.to_epsg() == 4326 and points_gdf["point_id"].is_unique
display(extent_summary.round(6))

,minimum_longitude,minimum_latitude,maximum_longitude,maximum_latitude
0,-82.606773,37.20715,-77.729781,40.623071


# Step 7 — Choose the predicate and CRS deliberately

`within` asks whether a point is strictly inside a polygon. It is the clearest primary predicate for assigning a point to an administrative area. A point exactly on a boundary may fail `within`, so the lesson retries only unmatched records with `intersects`. If an edge point touches multiple polygons, the code applies a stable identifier-based tie-break and records the event in QA.

The delivered files use `EPSG:4326`, but longitude/latitude degrees are not ideal analysis units. The assignment function transforms points and polygons to `EPSG:5070`, a projected CRS in meters, before joining. Spatial exports return to `EPSG:4326`.

In [6]:
# Step 8: Load and validate all four Census boundary layers.

expected_feature_counts = {
    "county": 55,
    "census_tract": 546,
    "incorporated_place": 231,
    "census_designated_place": 208,
}
boundaries = {}
boundary_qa_rows = []

for geography_level, path in BOUNDARY_PATHS.items():
    boundary = gpd.read_file(path, engine="pyogrio")
    required_fields = {"GEOID", "GEOIDFQ", "NAMELSAD", "geometry"}
    missing_fields = sorted(required_fields - set(boundary.columns))
    if missing_fields:
        raise ValueError(f"{path.name} is missing fields: {missing_fields}")

    null_or_empty = int(boundary.geometry.isna().sum() + boundary.geometry.is_empty.sum())
    invalid_mask = ~boundary.geometry.is_valid
    invalid_before = int(invalid_mask.sum())
    if invalid_before:
        boundary.loc[invalid_mask, "geometry"] = boundary.loc[invalid_mask, "geometry"].apply(make_valid)

    boundary = boundary.copy()
    boundary["geography_level"] = geography_level
    for field in ["GEOID", "GEOIDFQ", "NAMELSAD"]:
        boundary[field] = boundary[field].astype("string").str.strip()

    invalid_after = int((~boundary.geometry.is_valid).sum())
    duplicate_keys = int(boundary["GEOIDFQ"].duplicated().sum())
    assert boundary.crs is not None
    assert len(boundary) == expected_feature_counts[geography_level]
    assert null_or_empty == 0 and invalid_after == 0 and duplicate_keys == 0

    boundaries[geography_level] = boundary
    boundary_qa_rows.append({
        "geography_level": geography_level,
        "feature_count": len(boundary),
        "source_crs": str(boundary.crs),
        "null_or_empty_geometry_count": null_or_empty,
        "invalid_before_repair_count": invalid_before,
        "invalid_after_repair_count": invalid_after,
        "duplicate_key_count": duplicate_keys,
    })

boundary_qa = pd.DataFrame(boundary_qa_rows)
display(boundary_qa)

,geography_level,feature_count,source_crs,null_or_empty_geometry_count,invalid_before_repair_count,invalid_after_repair_count,duplicate_key_count
0,county,55,EPSG:4326,0,0,0,0
1,census_tract,546,EPSG:4326,0,0,0,0
2,incorporated_place,231,EPSG:4326,0,0,0,0
3,census_designated_place,208,EPSG:4326,0,0,0,0


In [7]:
# Step 9: Define a reusable point-in-polygon assignment function.

JOIN_CRS = "EPSG:5070"
BOUNDARY_FIELDS = ["geography_level", "GEOID", "GEOIDFQ", "NAMELSAD"]
POINT_FIELDS = [*point_fields, "geometry"]


def choose_one_match(joined: gpd.GeoDataFrame, method: str) -> tuple[gpd.GeoDataFrame, int]:
    '''Keep one deterministic polygon per point and count ambiguous records.'''
    matched = joined.loc[joined["index_right"].notna()].copy()
    if matched.empty:
        matched["assignment_method"] = pd.Series(dtype="string")
        return matched, 0
    match_counts = matched.groupby("point_id").size()
    ambiguous_count = int(match_counts.gt(1).sum())
    matched = matched.sort_values(["point_id", "GEOIDFQ", "boundary_row_id"])
    matched = matched.drop_duplicates("point_id", keep="first")
    matched["assignment_method"] = method
    return matched, ambiguous_count


def assign_points(points: gpd.GeoDataFrame, boundary: gpd.GeoDataFrame, geography_level: str):
    '''Assign points with `within`, retry edges with `intersects`, and return QA.'''
    points_for_join = points[POINT_FIELDS].to_crs(JOIN_CRS)
    boundary_for_join = boundary[BOUNDARY_FIELDS + ["geometry"]].to_crs(JOIN_CRS).copy()
    boundary_for_join["boundary_row_id"] = range(1, len(boundary_for_join) + 1)
    assert points_for_join.crs == boundary_for_join.crs

    within_join = gpd.sjoin(points_for_join, boundary_for_join, how="left", predicate="within")
    within_matches, within_ambiguous = choose_one_match(within_join, "within")

    within_ids = set(within_matches["point_id"])
    unmatched = points_for_join.loc[~points_for_join["point_id"].isin(within_ids)].copy()
    fallback_join = gpd.sjoin(unmatched, boundary_for_join, how="left", predicate="intersects")
    fallback_matches, fallback_ambiguous = choose_one_match(fallback_join, "intersects_boundary_fallback")

    assigned = gpd.GeoDataFrame(
        pd.concat([within_matches, fallback_matches], ignore_index=True),
        geometry="geometry", crs=JOIN_CRS,
    )
    unassigned_count = len(points_for_join) - len(assigned)

    point_totals = (
        assigned.groupby(BOUNDARY_FIELDS, dropna=False)
        .agg(
            point_count=("point_id", "nunique"),
            industry_code_count=("industry_code", "nunique"),
            total_employee_count=("employee_count", "sum"),
            total_revenue=("revenue", lambda values: values.sum(min_count=1)),
            revenue_record_count=("revenue", "count"),
        )
        .reset_index()
    )
    full_boundary_totals = boundary[BOUNDARY_FIELDS].drop_duplicates().merge(
        point_totals, on=BOUNDARY_FIELDS, how="left", validate="one_to_one"
    )
    integer_fill_fields = ["point_count", "industry_code_count", "total_employee_count", "revenue_record_count"]
    full_boundary_totals[integer_fill_fields] = full_boundary_totals[integer_fill_fields].fillna(0).astype("int64")
    full_boundary_totals["total_revenue"] = full_boundary_totals["total_revenue"].round().astype("Int64")

    qa_row = {
        "geography_level": geography_level,
        "boundary_feature_count": len(boundary),
        "coordinate_valid_point_count": len(points_for_join),
        "assigned_point_count": len(assigned),
        "unassigned_point_count": unassigned_count,
        "within_assigned_count": len(within_matches),
        "fallback_assigned_count": len(fallback_matches),
        "ambiguous_records_resolved": within_ambiguous + fallback_ambiguous,
        "polygons_with_points": int(full_boundary_totals["point_count"].gt(0).sum()),
    }
    assert assigned["point_id"].is_unique
    assert len(assigned) + unassigned_count == len(points_for_join)
    assert int(full_boundary_totals["point_count"].sum()) == len(assigned)
    return assigned, full_boundary_totals, qa_row

In [8]:
# Step 10: Assign points to statewide counties and census tracts.

county_assigned, county_totals, county_qa = assign_points(
    points_gdf, boundaries["county"], "county"
)
tract_assigned, tract_totals, tract_qa = assign_points(
    points_gdf, boundaries["census_tract"], "census_tract"
)

# Counties and tracts cover the state, so these are strong reconciliation benchmarks.
for qa_row in [county_qa, tract_qa]:
    qa_row["coverage_expectation"] = "statewide_near_complete"
    qa_row["benchmark_status"] = (
        "PASS" if (qa_row["assigned_point_count"], qa_row["unassigned_point_count"]) == (62_544, 3)
        else "REVIEW"
    )

display(pd.DataFrame([county_qa, tract_qa]))

,geography_level,boundary_feature_count,coordinate_valid_point_count,assigned_point_count,unassigned_point_count,within_assigned_count,fallback_assigned_count,ambiguous_records_resolved,polygons_with_points,coverage_expectation,benchmark_status
0,county,55,62547,62544,3,62544,0,0,55,statewide_near_complete,PASS
1,census_tract,546,62547,62544,3,62544,0,0,546,statewide_near_complete,PASS


In [9]:
# Step 11: Assign points to incorporated places.

incorporated_place_assigned, incorporated_place_totals, incorporated_place_qa = assign_points(
    points_gdf, boundaries["incorporated_place"], "incorporated_place"
)
incorporated_place_qa["coverage_expectation"] = "partial_settlement_coverage"
incorporated_place_qa["benchmark_status"] = (
    "PASS" if (incorporated_place_qa["assigned_point_count"], incorporated_place_qa["unassigned_point_count"])
    == (37_639, 24_908) else "REVIEW"
)

display(pd.DataFrame([incorporated_place_qa]))
display(incorporated_place_totals.sort_values("point_count", ascending=False).head(10))

,geography_level,boundary_feature_count,coordinate_valid_point_count,assigned_point_count,unassigned_point_count,within_assigned_count,fallback_assigned_count,ambiguous_records_resolved,polygons_with_points,coverage_expectation,benchmark_status
0,incorporated_place,231,62547,37639,24908,37639,0,0,230,partial_settlement_coverage,PASS


,geography_level,GEOID,GEOIDFQ,NAMELSAD,point_count,industry_code_count,total_employee_count,total_revenue,revenue_record_count
41,incorporated_place,5414600,1600000US5414600,Charleston city,3894,403,75959,9471370000,2416
95,incorporated_place,5439460,1600000US5439460,Huntington city,2218,377,33127,4989374000,1510
217,incorporated_place,5486452,1600000US5486452,Wheeling city,1883,337,23365,4832607000,1238
148,incorporated_place,5462140,1600000US5462140,Parkersburg city,1761,328,20594,3926975000,1187
132,incorporated_place,5455756,1600000US5455756,Morgantown city,1663,318,32520,2486528000,1064
13,incorporated_place,5405332,1600000US5405332,Beckley city,1324,278,15384,2522557000,918
119,incorporated_place,5452060,1600000US5452060,Martinsburg city,1129,267,12722,1194574000,740
62,incorporated_place,5426452,1600000US5426452,Fairmont city,914,243,9501,950762000,576
190,incorporated_place,5475292,1600000US5475292,South Charleston city,853,244,12140,2738673000,597
45,incorporated_place,5415628,1600000US5415628,Clarksburg city,827,240,9607,1606158000,516


In [10]:
# Step 12: Assign points to Census designated places (CDPs).

cdp_assigned, cdp_totals, cdp_qa = assign_points(
    points_gdf, boundaries["census_designated_place"], "census_designated_place"
)
cdp_qa["coverage_expectation"] = "partial_settlement_coverage"
cdp_qa["benchmark_status"] = (
    "PASS" if (cdp_qa["assigned_point_count"], cdp_qa["unassigned_point_count"])
    == (4_981, 57_566) else "REVIEW"
)

display(pd.DataFrame([cdp_qa]))
display(cdp_totals.sort_values("point_count", ascending=False).head(10))

,geography_level,boundary_feature_count,coordinate_valid_point_count,assigned_point_count,unassigned_point_count,within_assigned_count,fallback_assigned_count,ambiguous_records_resolved,polygons_with_points,coverage_expectation,benchmark_status
0,census_designated_place,208,62547,4981,57566,4981,0,0,197,partial_settlement_coverage,PASS


,geography_level,GEOID,GEOIDFQ,NAMELSAD,point_count,industry_code_count,total_employee_count,total_revenue,revenue_record_count
206,census_designated_place,5479545,1600000US5479545,Teays Valley CDP,642,196,5772,823323000,487
68,census_designated_place,5414775,1600000US5414775,Cheat Lake CDP,243,114,2086,289199000,192
89,census_designated_place,5426428,1600000US5426428,Fairlea CDP,235,106,2245,260427000,162
97,census_designated_place,5419108,1600000US5419108,Cross Lanes CDP,231,105,1937,272660000,168
162,census_designated_place,5462488,1600000US5462488,Pea Ridge CDP,173,86,1422,287729000,143
152,census_designated_place,5456342,1600000US5456342,Mount Gay-Shamrock CDP,166,84,1691,301760000,118
146,census_designated_place,5440204,1600000US5440204,Inwood CDP,125,72,951,90820000,87
66,census_designated_place,5410420,1600000US5410420,Brookhaven CDP,121,75,727,206891000,82
36,census_designated_place,5465836,1600000US5465836,Prosperity CDP,118,68,1149,198010000,91
179,census_designated_place,5474356,1600000US5474356,Sissonville CDP,111,63,1022,260337000,79


# Step 13 — Interpret unmatched points according to geography coverage

County and tract nonmatches are review items because those layers cover the state. Incorporated-place and CDP nonmatches are expected because many locations are rural or otherwise outside a named settlement boundary.

Do not merge incorporated places and CDPs by name. They are separate Census feature classes, and stable `GEOIDFQ` identifiers are safer than labels for downstream joins.

In [11]:
# Step 14: Assemble one QA table for all four geography levels.

assignment_qa = pd.DataFrame([county_qa, tract_qa, incorporated_place_qa, cdp_qa])
assert assignment_qa["benchmark_status"].eq("PASS").all()
display(assignment_qa)

,geography_level,boundary_feature_count,coordinate_valid_point_count,assigned_point_count,unassigned_point_count,within_assigned_count,fallback_assigned_count,ambiguous_records_resolved,polygons_with_points,coverage_expectation,benchmark_status
0,county,55,62547,62544,3,62544,0,0,55,statewide_near_complete,PASS
1,census_tract,546,62547,62544,3,62544,0,0,546,statewide_near_complete,PASS
2,incorporated_place,231,62547,37639,24908,37639,0,0,230,partial_settlement_coverage,PASS
3,census_designated_place,208,62547,4981,57566,4981,0,0,197,partial_settlement_coverage,PASS


In [12]:
# Step 15: Combine all geography identifiers into one point-assignment table.

def assignment_keys(assigned, prefix):
    '''Rename one assignment result so it can be joined to the point table.'''
    return assigned[["point_id", "GEOID", "GEOIDFQ", "NAMELSAD", "assignment_method"]].rename(columns={
        "GEOID": f"{prefix}_geoid",
        "GEOIDFQ": f"{prefix}_geoidfq",
        "NAMELSAD": f"{prefix}_name",
        "assignment_method": f"{prefix}_assignment_method",
    })


point_assignments = coordinate_valid_df[point_fields].copy()
for assigned, prefix in [
    (county_assigned, "county"),
    (tract_assigned, "tract"),
    (incorporated_place_assigned, "incorporated_place"),
    (cdp_assigned, "census_designated_place"),
]:
    point_assignments = point_assignments.merge(
        assignment_keys(assigned, prefix), on="point_id", how="left", validate="one_to_one"
    )
    point_assignments[f"{prefix}_match_status"] = point_assignments[f"{prefix}_geoidfq"].notna().map(
        {True: "matched", False: "unmatched"}
    )

assert len(point_assignments) == len(coordinate_valid_df)
assert point_assignments["point_id"].is_unique
display(point_assignments[[
    "county_match_status", "tract_match_status",
    "incorporated_place_match_status", "census_designated_place_match_status",
]].value_counts().rename("point_count").reset_index())

,county_match_status,tract_match_status,incorporated_place_match_status,census_designated_place_match_status,point_count
0,matched,matched,matched,unmatched,37639
1,matched,matched,unmatched,unmatched,19924
2,matched,matched,unmatched,matched,4981
3,unmatched,unmatched,unmatched,unmatched,3


In [13]:
# Step 16: Aggregate employees and revenue by geography and industry code.

def summarize_by_industry(assigned: gpd.GeoDataFrame) -> pd.DataFrame:
    '''Return one row per geography/industry code, sorted for review and export.'''
    summary = (
        assigned.groupby(
            ["geography_level", "GEOID", "GEOIDFQ", "NAMELSAD",
             "industry_code", "industry_sector", "industry_description"],
            dropna=False,
        )
        .agg(
            point_count=("point_id", "nunique"),
            total_employee_count=("employee_count", "sum"),
            total_revenue=("revenue", lambda values: values.sum(min_count=1)),
            revenue_record_count=("revenue", "count"),
            revenue_missing_count=("revenue", lambda values: int(values.isna().sum())),
        )
        .reset_index()
    )
    summary["total_revenue"] = summary["total_revenue"].round().astype("Int64")
    return summary.sort_values(["GEOIDFQ", "industry_code"], kind="stable").reset_index(drop=True)


industry_summaries = {
    "county": summarize_by_industry(county_assigned),
    "census_tract": summarize_by_industry(tract_assigned),
    "incorporated_place": summarize_by_industry(incorporated_place_assigned),
    "census_designated_place": summarize_by_industry(cdp_assigned),
}

industry_output_qa = pd.DataFrame([
    {
        "geography_level": level,
        "output_rows": len(summary),
        "geographies_represented": summary["GEOIDFQ"].nunique(),
        "industry_codes_represented": summary["industry_code"].nunique(),
        "point_count_total": int(summary["point_count"].sum()),
        "employee_count_total": int(summary["total_employee_count"].sum()),
        "revenue_total_for_nonmissing_records": int(summary["total_revenue"].sum(min_count=1)),
        "revenue_records": int(summary["revenue_record_count"].sum()),
    }
    for level, summary in industry_summaries.items()
])
display(industry_output_qa)
display(industry_summaries["county"].head(10))

,geography_level,output_rows,geographies_represented,industry_codes_represented,point_count_total,employee_count_total,revenue_total_for_nonmissing_records,revenue_records
0,county,12056,55,787,62544,698343,104326766000,40620
1,census_tract,33733,546,787,62544,698343,104326766000,40620
2,incorporated_place,13737,230,690,37639,453553,61429237000,24036
3,census_designated_place,3218,197,440,4981,43294,6900709000,3280


,geography_level,GEOID,GEOIDFQ,NAMELSAD,industry_code,industry_sector,industry_description,point_count,total_employee_count,total_revenue,revenue_record_count,revenue_missing_count
0,county,54001,0500000US54001,Barbour County,112519,"Agriculture, Forestry, Fishing and Hunting",Other Aquaculture,1,1,41000,1,0
1,county,54001,0500000US54001,Barbour County,112920,"Agriculture, Forestry, Fishing and Hunting",Horses and Other Equine Production,1,2,88000,1,0
2,county,54001,0500000US54001,Barbour County,115112,"Agriculture, Forestry, Fishing and Hunting","Soil Preparation, Planting, and Cultivating",1,6,945000,1,0
3,county,54001,0500000US54001,Barbour County,221122,"Utility Services: Power, Gas, Steam, Water, an...",Electric Power Distribution,1,3,1515000,1,0
4,county,54001,0500000US54001,Barbour County,221310,"Utility Services: Power, Gas, Steam, Water, an...",Water Supply and Irrigation Systems,1,6,733000,1,0
5,county,54001,0500000US54001,Barbour County,236115,Construction,New Single-Family Housing Construction (except...,2,12,1470000,2,0
6,county,54001,0500000US54001,Barbour County,236220,Construction,Commercial and Institutional Building Construc...,3,20,2451000,3,0
7,county,54001,0500000US54001,Barbour County,237310,Construction,"Road/Street, Highway and Bridge Construction",1,31,<NA>,0,1
8,county,54001,0500000US54001,Barbour County,237990,Construction,Other Heavy and Civil Engineering Construction,1,2,548000,1,0
9,county,54001,0500000US54001,Barbour County,238220,Construction,"Plumbing, Heating, and Air-Conditioning Contra...",5,47,4124000,5,0


# Step 17 — Read aggregate measures carefully

Each industry output is sorted first by Census geography identifier and then by industry code. The totals mean:

- `point_count`: distinct point records in that geography/industry group;
- `total_employee_count`: sum of the employee estimates for those records;
- `total_revenue`: sum of available revenue estimates; and
- `revenue_record_count` / `revenue_missing_count`: coverage fields showing how many records did or did not contribute to revenue.

A blank `total_revenue` means every record in that row lacked a revenue value. It is intentionally not converted to zero. These are establishment-location aggregates, not resident-worker employment statistics.

In [14]:
# Step 18: Join county totals back to polygons for GIS export and mapping.

county_summary_gdf = boundaries["county"][["GEOID", "GEOIDFQ", "NAMELSAD", "boundary_year", "geometry"]].merge(
    county_totals[["GEOIDFQ", "point_count", "industry_code_count", "total_employee_count",
                   "total_revenue", "revenue_record_count"]],
    on="GEOIDFQ", how="left", validate="one_to_one",
)
fill_fields = ["point_count", "industry_code_count", "total_employee_count", "revenue_record_count"]
county_summary_gdf[fill_fields] = county_summary_gdf[fill_fields].fillna(0).astype("int64")
assert len(county_summary_gdf) == 55
assert int(county_summary_gdf["point_count"].sum()) == county_qa["assigned_point_count"]
display(county_summary_gdf.drop(columns="geometry").sort_values("point_count", ascending=False).head(10))

,GEOID,GEOIDFQ,NAMELSAD,boundary_year,point_count,industry_code_count,total_employee_count,total_revenue,revenue_record_count
51,54039,0500000US54039,Kanawha County,2024,7599,510,119070,18447025000,4912
7,54061,0500000US54061,Monongalia County,2024,4262,438,62676,7418657000,3033
4,54011,0500000US54011,Cabell County,2024,3637,445,51195,8540095000,2573
12,54107,0500000US54107,Wood County,2024,3257,420,36832,6647840000,2281
39,54003,0500000US54003,Berkeley County,2024,3057,410,30906,3536881000,2163
42,54081,0500000US54081,Raleigh County,2024,2998,398,33487,5785568000,2053
16,54033,0500000US54033,Harrison County,2024,2668,377,31859,4654761000,1817
49,54069,0500000US54069,Ohio County,2024,2391,375,32948,7504614000,1589
24,54055,0500000US54055,Mercer County,2024,2100,344,21238,4074659000,1395
36,54049,0500000US54049,Marion County,2024,1914,345,18531,2584580000,1271


In [15]:
# Step 19: Build a lightweight visual-QA map.

# Simplify only a display copy in projected meters; keep analytic geometry unchanged.
county_display = county_summary_gdf.to_crs(JOIN_CRS).copy()
county_display["geometry"] = county_display.geometry.simplify(500, preserve_topology=True)
county_display = county_display.to_crs("EPSG:4326")

minimum_count = int(county_display["point_count"].min())
maximum_count = int(county_display["point_count"].max())
point_colormap = linear.YlGnBu_09.scale(minimum_count, maximum_count)
point_colormap.caption = "Point records assigned to county"

county_map = folium.Map(location=[38.6, -80.6], zoom_start=7, tiles=None, control_scale=True)
folium.GeoJson(
    county_display[["NAMELSAD", "point_count", "total_employee_count", "total_revenue", "geometry"]].to_json(drop_id=True),
    name="County point summary",
    style_function=lambda feature: {
        "fillColor": point_colormap(feature["properties"]["point_count"]),
        "color": "#475569", "weight": 0.8, "fillOpacity": 0.78,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NAMELSAD", "point_count", "total_employee_count", "total_revenue"],
        aliases=["County", "Point count", "Employees", "Revenue with available values"],
        localize=True,
    ),
).add_to(county_map)
point_colormap.add_to(county_map)
min_x, min_y, max_x, max_y = county_display.total_bounds
county_map.fit_bounds([[min_y, min_x], [max_y, max_x]])
county_map

In [16]:
# Step 20: Export assignments, geography totals, industry summaries, QA, geometry, and the map.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_paths = {
    "point_assignments": OUTPUT_DIR / "point_assignments.csv",
    "county_summary": OUTPUT_DIR / "county_point_summary.csv",
    "tract_summary": OUTPUT_DIR / "census_tract_point_summary.csv",
    "incorporated_place_summary": OUTPUT_DIR / "incorporated_place_point_summary.csv",
    "cdp_summary": OUTPUT_DIR / "census_designated_place_point_summary.csv",
    "spatial_qa": OUTPUT_DIR / "spatial_join_qa.csv",
    "industry_qa": OUTPUT_DIR / "industry_summary_qa.csv",
    "county_geojson": OUTPUT_DIR / "county_point_summary.geojson",
    "county_map": OUTPUT_DIR / "county_point_summary_map.html",
}
industry_output_paths = {
    "county": OUTPUT_DIR / "county_industry_employee_revenue.csv",
    "census_tract": OUTPUT_DIR / "census_tract_industry_employee_revenue.csv",
    "incorporated_place": OUTPUT_DIR / "incorporated_place_industry_employee_revenue.csv",
    "census_designated_place": OUTPUT_DIR / "census_designated_place_industry_employee_revenue.csv",
}

point_assignments.to_csv(output_paths["point_assignments"], index=False, encoding="utf-8")
county_totals.to_csv(output_paths["county_summary"], index=False, encoding="utf-8")
tract_totals.to_csv(output_paths["tract_summary"], index=False, encoding="utf-8")
incorporated_place_totals.to_csv(output_paths["incorporated_place_summary"], index=False, encoding="utf-8")
cdp_totals.to_csv(output_paths["cdp_summary"], index=False, encoding="utf-8")

for level, summary in industry_summaries.items():
    summary.to_csv(industry_output_paths[level], index=False, encoding="utf-8")

qa_export = assignment_qa.copy()
qa_export.insert(0, "generated_utc", datetime.now(timezone.utc).replace(microsecond=0).isoformat())
qa_export["input_point_rows"] = len(points_df)
qa_export["coordinate_issue_rows"] = len(coordinate_issue_df)
qa_export.to_csv(output_paths["spatial_qa"], index=False, encoding="utf-8")
industry_output_qa.to_csv(output_paths["industry_qa"], index=False, encoding="utf-8")
county_summary_gdf.to_crs("EPSG:4326").to_file(
    output_paths["county_geojson"], driver="GeoJSON", engine="pyogrio"
)
county_map.save(output_paths["county_map"])

for path in [*output_paths.values(), *industry_output_paths.values()]:
    print(f"Wrote {path.name}")

Wrote point_assignments.csv
Wrote county_point_summary.csv
Wrote census_tract_point_summary.csv
Wrote incorporated_place_point_summary.csv
Wrote census_designated_place_point_summary.csv
Wrote spatial_join_qa.csv
Wrote industry_summary_qa.csv
Wrote county_point_summary.geojson
Wrote county_point_summary_map.html
Wrote county_industry_employee_revenue.csv
Wrote census_tract_industry_employee_revenue.csv
Wrote incorporated_place_industry_employee_revenue.csv
Wrote census_designated_place_industry_employee_revenue.csv


In [17]:
# Step 21: Read every output back and verify row counts, sorting, and reconciliation.

assignment_readback = pd.read_csv(output_paths["point_assignments"], dtype={"point_id": "string"})
qa_readback = pd.read_csv(output_paths["spatial_qa"])
county_geojson_readback = gpd.read_file(output_paths["county_geojson"], engine="pyogrio")

summary_expected_rows = {
    "county_summary": 55,
    "tract_summary": 546,
    "incorporated_place_summary": 231,
    "cdp_summary": 208,
}
validation_rows = []
assert len(assignment_readback) == 62_547 and assignment_readback["point_id"].is_unique
validation_rows.append({"output": output_paths["point_assignments"].name, "rows": len(assignment_readback), "status": "PASS"})

for key, expected_rows in summary_expected_rows.items():
    frame = pd.read_csv(output_paths[key], dtype={"GEOID": "string", "GEOIDFQ": "string"})
    assert len(frame) == expected_rows and frame["GEOIDFQ"].is_unique
    validation_rows.append({"output": output_paths[key].name, "rows": len(frame), "status": "PASS"})

for level, path in industry_output_paths.items():
    frame = pd.read_csv(path, dtype={"GEOID": "string", "GEOIDFQ": "string", "industry_code": "string"})
    sorted_keys = frame[["GEOIDFQ", "industry_code"]].sort_values(
        ["GEOIDFQ", "industry_code"], kind="stable"
    ).reset_index(drop=True)
    assert frame[["GEOIDFQ", "industry_code"]].reset_index(drop=True).equals(sorted_keys)
    assert int(frame["point_count"].sum()) == int(
        qa_readback.loc[qa_readback["geography_level"].eq(level), "assigned_point_count"].iloc[0]
    )
    validation_rows.append({"output": path.name, "rows": len(frame), "status": "PASS"})

assert len(qa_readback) == 4
assert len(county_geojson_readback) == 55 and county_geojson_readback.crs.to_epsg() == 4326
assert county_geojson_readback.geometry.notna().all() and county_geojson_readback.geometry.is_valid.all()
assert output_paths["county_map"].exists() and output_paths["county_map"].stat().st_size > 0
validation_rows.extend([
    {"output": output_paths["spatial_qa"].name, "rows": len(qa_readback), "status": "PASS"},
    {"output": output_paths["county_geojson"].name, "rows": len(county_geojson_readback), "status": "PASS"},
    {"output": output_paths["county_map"].name, "rows": 1, "status": "PASS"},
])
display(pd.DataFrame(validation_rows))

,output,rows,status
0,point_assignments.csv,62547,PASS
1,county_point_summary.csv,55,PASS
2,census_tract_point_summary.csv,546,PASS
3,incorporated_place_point_summary.csv,231,PASS
4,census_designated_place_point_summary.csv,208,PASS
5,county_industry_employee_revenue.csv,12056,PASS
6,census_tract_industry_employee_revenue.csv,33733,PASS
7,incorporated_place_industry_employee_revenue.csv,13737,PASS
8,census_designated_place_industry_employee_reve...,3218,PASS
9,spatial_join_qa.csv,4,PASS


# Step 22 — Generalize the workflow

The reusable pattern is complete:

1. validate coordinates and measures;
2. create point geometry with the correct source CRS;
3. validate polygon geometry and stable identifiers;
4. transform both layers to a shared join CRS;
5. use `within`, then a limited `intersects` fallback;
6. preserve and interpret unmatched records according to polygon coverage;
7. aggregate by geography identifier and industry code;
8. carry measure-coverage fields beside summed values; and
9. read generated outputs back before handoff.

To adapt the lesson, replace the point CSV or add another boundary GeoJSON with `GEOID`, `GEOIDFQ`, `NAMELSAD`, and geometry. Avoid joining on names. Decide in advance whether the selected polygons are exhaustive, and never treat missing revenue as zero without an explicit business rule.

Because this package uses real locations and measures, review sharing permissions before substituting another point dataset.